# Assignment 7 - Mode Collapse (Simple Version)

Goal: break a GAN, then fix it.

1. Run A (broken) -> saves `collapsed.png` - all images look the same.
2. Run B (fixed) -> saves `improved.png` - images look different.

How to run in Colab: Runtime > Change runtime to GPU > Run all. Each run takes ~5 min.

In [ ]:
# Step 1: Settings. Only these 5 things change between runs.
import random, numpy as np, torch, matplotlib.pyplot as plt
import torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T, torchvision.utils as vutils

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

NZ = 100  # random noise size
EPOCHS = 15
BATCH = 128

# Broken: small brain, no BatchNorm, learning rate too big
BROKEN = {'hidden': 32, 'use_bn': False, 'lr_g': 0.01, 'lr_d': 0.01, 'label': 1.0}
# Fixed: bigger brain, BatchNorm back, small learning rates, soft label 0.9
FIXED  = {'hidden': 256, 'use_bn': True, 'lr_g': 0.0002, 'lr_d': 0.0001, 'label': 0.9}

# Same noise for both runs, so we can compare fairly
fixed_noise = torch.randn(64, NZ, device=device)

In [ ]:
# Step 2: Data - MNIST handwritten digits
loader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST('./data', train=True, download=True,
        transform=T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])),
    batch_size=BATCH, shuffle=True, num_workers=2)
print('batches per epoch:', len(loader))

In [ ]:
# Step 3: Two small models. Plain Linear layers only.
class G(nn.Module):  # Generator: noise -> image
    def __init__(self, hidden, use_bn):
        super().__init__()
        self.fc1 = nn.Linear(NZ, hidden)
        self.bn = nn.BatchNorm1d(hidden) if use_bn else nn.Identity()
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 28*28)
    def forward(self, z):
        x = torch.relu(self.bn(self.fc1(z)))
        x = torch.relu(self.fc2(x))
        return torch.tanh(self.fc3(x))  # output in [-1, 1]

class D(nn.Module):  # Discriminator: image -> real or fake (0 to 1)
    def __init__(self):
        super().__init__()
        self.m = nn.Sequential(nn.Linear(28*28, 256), nn.LeakyReLU(0.2),
                               nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, img):
        return self.m(img.view(img.size(0), -1)).view(-1)

In [ ]:
# Step 4: One training function used by both runs
def train(cfg, save_as):
    torch.manual_seed(SEED)  # same start for both runs
    netG, netD = G(cfg['hidden'], cfg['use_bn']).to(device), D().to(device)
    optG = optim.Adam(netG.parameters(), lr=cfg['lr_g'], betas=(0.5, 0.999))
    optD = optim.Adam(netD.parameters(), lr=cfg['lr_d'], betas=(0.5, 0.999))
    loss_fn = nn.BCELoss()
    Gl, Dl = [], []
    for ep in range(1, EPOCHS+1):
        for real, _ in loader:
            b = real.size(0); real = real.to(device)
            # train D: real=label, fake=0
            netD.zero_grad()
            fake = netG(torch.randn(b, NZ, device=device))
            lossD = loss_fn(netD(real), torch.full((b,), cfg['label'], device=device)) + loss_fn(netD(fake.detach()), torch.zeros(b, device=device))
            lossD.backward(); optD.step()
            # train G: wants D to say 1 (or 0.9)
            netG.zero_grad()
            lossG = loss_fn(netD(fake), torch.full((b,), cfg['label'], device=device))
            lossG.backward(); optG.step()
            Gl.append(lossG.item()); Dl.append(lossD.item())
        print(f"epoch {ep}/{EPOCHS} lossG {lossG.item():.3f} lossD {lossD.item():.3f}")
        if ep in [1, 5, 10, 15]:  # show progress
            with torch.no_grad():
                imgs = netG(fixed_noise).view(-1, 1, 28, 28).cpu()
            plt.figure(figsize=(4,4)); plt.axis('off'); plt.title(f"epoch {ep}")
            plt.imshow(vutils.make_grid(imgs, nrow=8, normalize=True).permute(1,2,0)); plt.show()
    with torch.no_grad():
        final = netG(fixed_noise).view(-1, 1, 28, 28).cpu()
    vutils.save_image(final, save_as, nrow=8, normalize=True)
    print('saved', save_as)
    return Gl, Dl

## Run A - Broken (this should collapse)

In [ ]:
Gl_bad, Dl_bad = train(BROKEN, 'collapsed.png')
from PIL import Image; display(Image.open('collapsed.png'))

## Run B - Fixed (this should look better)

In [ ]:
Gl_good, Dl_good = train(FIXED, 'improved.png')
from PIL import Image; display(Image.open('improved.png'))

In [ ]:
# Compare + loss curves
from PIL import Image
fig, ax = plt.subplots(1, 2, figsize=(9,4))
ax[0].imshow(Image.open('collapsed.png')); ax[0].set_title('A - broken'); ax[0].axis('off')
ax[1].imshow(Image.open('improved.png')); ax[1].set_title('B - fixed'); ax[1].axis('off')
plt.show()
plt.plot(Gl_bad, label='G bad'); plt.plot(Dl_bad, label='D bad')
plt.plot(Gl_good, label='G good'); plt.plot(Dl_good, label='D good')
plt.legend(); plt.title('Loss curves'); plt.show()

## Write-up (copy this to your assignment)

1. **What caused poor performance?** I removed BatchNorm, made the generator very small (32 units), and set the learning rate to 0.01, which is far too big. Training became unstable and the generator just repeated the few outputs that fooled the discriminator.

2. **What did the images look like?** In `collapsed.png` almost all 64 images are the same blurry digit, even though the noise inputs are different. This is mode collapse.

3. **Which technique fixed it?** Label smoothing (real label 0.9 instead of 1.0) plus a slower discriminator (G lr 0.0002, D lr 0.0001), with BatchNorm and full size (256) restored.

4. **Did it improve? Why?** Yes. In `improved.png` the digits are varied and clear. The soft label stops the discriminator from getting overconfident, and the slower discriminator gives the generator time to learn, so it covers many digits instead of one.